In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


**CWRU dataset:** Smith, W. A., & Randall, R. B. (2015). Rolling element bearing diagnostics using the Case Western Reserve University data: A benchmark study. *Mechanical Systems and Signal Processing*, 64, 100–131.


# Feature Practice — CWRU 베어링 내륜 결함 진단

FFT · Filtering · Time-Frequency · Envelope 도구를 **하나의 실제 데이터 (CWRU 베어링 가속도 신호)** 위에서 파이프라인으로 연결한다.

**목표**
- 정상 신호 (`v_n`) 와 내륜 결함 (inner race, IR) 신호 (`v_f`) 를 비교한다.
- 시간 영역 → 주파수 영역 → 공진 대역 격리 → 포락선 복조 (demodulation) 순서로 단계를 따라간다.
- 각 단계에서 정상/결함 차이가 어떻게 부각되는지 관찰하고, 마지막에 **특징 벡터 (feature vector)** 로 정량화한다.

**데이터 출처**: Case Western Reserve University Bearing Data Center (1772 rpm, 12 kHz 샘플링, Drive-End 가속도).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils


## 실습 1. 베어링 특성 주파수 계산

베어링 결함이 발생하면, **회전 1바퀴당 결함부가 하중 구간을 지나가는 횟수**가 이론적으로 결정된다. CWRU Drive-End 베어링(SKF 6205)의 기하학적 상수는 다음과 같다.

| 결함 위치 | 특성 주파수 (shaft frequency 기준 배수) |
| --- | --- |
| 내륜 (Inner Race, BPFI) | 5.4152 |
| 외륜 (Outer Race, BPFO) | 3.5848 |

축 회전 속도 $f_{shaft} = \text{rpm}/60$을 곱하면 절대 주파수(Hz)가 나온다. 이 값이 **포락선 스펙트럼에서 기대되는 피크 위치**가 된다.


In [ ]:
rpm = 1772
F_shaft = rpm / 60
print('Shaft frequency:', F_shaft, 'Hz')

# Drive-End 베어링 특성 주파수
DE_BPFI = 5.4152 * F_shaft
DE_BPFO = 3.5848 * F_shaft
print('DE Inner Race Fault Frequency (BPFI):', DE_BPFI, 'Hz')
print('DE Outer Race Fault Frequency (BPFO):', DE_BPFO, 'Hz')

# Fan-End 베어링 (참고)
FE_BPFI = 4.9469 * F_shaft
FE_BPFO = 3.0530 * F_shaft
print('FE Inner Race Fault Frequency:', FE_BPFI, 'Hz')
print('FE Outer Race Fault Frequency:', FE_BPFO, 'Hz')


**관찰**
- Shaft 주파수는 약 **29.5 Hz**, DE 내륜 결함 주파수 **BPFI ≈ 160 Hz**이다.
- 이 노트북에서는 내륜 결함 데이터만 다루므로, 이후 `DE_BPFI` 및 그 배음(2배, 3배, …)이 포락선 스펙트럼의 **진단 지표**가 된다.
- 단, **슬립(slip)** 때문에 실제 피크는 이론값에서 ±1~2% 벗어나는 게 정상이다.


---


## 실습 2. 신호 로드와 파형 비교

CSV 파일 두 개를 읽어 1-D 배열로 만든다. 컬럼 0은 샘플 인덱스, 컬럼 1이 가속도(g) 값이다.

- `data/data_normal.csv` → 정상 베어링 (`v_n`)
- `data/data_fault_DE_IR.csv` → 내륜 결함 (`v_f`)

샘플링 주파수는 **12 kHz** (CWRU 규격). 두 신호는 길이가 동일(10초)하므로 시간축을 공유할 수 있다.


In [ ]:
fs = 12000

data = np.array(pd.read_csv('./data/data_normal.csv'))
v_n = data[:, 1]

data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
v_f = data[:, 1]

# 시간 벡터: arange(N)/fs 로 정확히 N 개 샘플에 매핑 (오프셋 없이)
t_n = np.arange(len(v_n)) / fs
t_f = np.arange(len(v_f)) / fs

# 파형 비교 (Fault peak 기준 자동 스케일)
y_lim = np.max(np.abs(v_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (g)')
plt.ylim([-y_lim, y_lim])
plt.legend()
plt.tight_layout()
plt.show()

print(f'RMS  Normal = {np.sqrt(np.mean(v_n**2)):.4f} g')
print(f'RMS  Fault  = {np.sqrt(np.mean(v_f**2)):.4f} g')
print(f'Peak Normal = {np.max(np.abs(v_n)):.4f} g')
print(f'Peak Fault  = {np.max(np.abs(v_f)):.4f} g')


**관찰**
- 결함 신호는 정상 대비 **RMS 약 1.2배**, **Peak 약 2배** 수준으로 커진다.
- RMS보다 Peak가 더 크게 벌어진다는 점이 중요하다 — 결함이 **임펄스성(peak-y)** 이라는 뜻이다.
- 하지만 시간 파형만 봐서는 "진폭이 조금 커졌다" 외에 정보가 부족하다. 주파수 영역으로 가자.


---


## 실습 3. 스펙트럼 비교

두 신호의 단측 진폭 스펙트럼을 구해 나란히 비교한다. 베어링 결함의 임펄스는 **구조물 공진 대역(수 kHz)** 을 주기적으로 때려 그 대역 에너지를 키운다. 따라서 저주파가 아니라 **고주파 공진대**를 먼저 확인해야 한다.


In [ ]:
f_n, A_n = utils.fft(v_n - np.mean(v_n), fs)
f_f, A_f = utils.fft(v_f - np.mean(v_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_n, A_n, color='C0', label='Normal', alpha=0.9)
plt.plot(f_f, A_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, fs / 2])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 결함 신호는 **약 3~6 kHz 사이**의 브로드밴드 대역에서 정상 대비 **수 배**로 올라온다 — 이 구간이 구조물 공진대이다.
- 저주파(<500 Hz)에는 BPFI가 이론적으로 존재해야 하지만, 여기서는 거의 보이지 않는다. 직접 FFT로는 **결함 진단 정보가 고주파 캐리어에 실려** 있어서, 저주파에서는 묻힌다.
- 해결책: **공진 대역만 잘라내서 (bandpass)** → **포락선을 구해서 (Hilbert)** → **다시 FFT**하면 저주파 BPFI가 drift 없이 살아난다. 다음 단계로.


---


## 실습 4. 대역통과 필터링 — 공진 대역 격리

실습 3에서 확인한 공진대 **[3500, 5500] Hz**만 통과시킨다. Butterworth 4차 zero-phase (`filtfilt`)이므로 위상 왜곡은 없다.

필터링된 신호는 **공진 주파수로 진동하는 캐리어(carrier)** 가 **결함 임펄스의 리듬(BPFI, ~160 Hz)** 으로 진폭 변조된 형태가 된다. 이 AM 구조에서 변조 정보를 꺼내는 게 다음 단계(포락선 복조).


In [ ]:
f_low, f_high = 3500, 5500

v_filter_n = utils.filtering_zerophase(v_n, fs, 'band', f_low=f_low, f_high=f_high)
v_filter_f = utils.filtering_zerophase(v_f, fs, 'band', f_low=f_low, f_high=f_high)

# 자동 스케일 (fault 기준)
y_lim_filter = np.max(np.abs(v_filter_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_filter_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_filter_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Filtered amplitude (g)')
plt.ylim([-y_lim_filter, y_lim_filter])
plt.title(f'Bandpass [{f_low}, {f_high}] Hz')
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 필터링 후 **Fault 파형의 임펄스 구조가 선명해진다** — 공진 캐리어가 BPFI 주기로 amplitude modulation 되어 있음을 육안으로도 볼 수 있다.
- Normal은 같은 대역에서 거의 평평한 잡음 수준이다.
- 이제 이 Fault 신호의 **상단 포락선(envelope)** 만 뽑아내면, 고주파 캐리어는 지워지고 **BPFI 리듬만** 남는다.


---


## 실습 5. 포락선 스펙트럼 — BPFI 확인

**Hilbert envelope**로 필터링된 신호의 순시 진폭을 구한 뒤, DC 성분을 빼고 FFT 한다 (`utils.hilbert_envelope`).

포락선 스펙트럼의 저주파 구간 (0~500 Hz) 에서 다음을 기대한다.
- **BPFI ≈ 160 Hz** 근처에 뚜렷한 피크
- 그 **2배(≈320 Hz), 3배(≈480 Hz)** 배음(harmonic)
- Shaft 주파수(≈30 Hz) 근처의 사이드밴드 (내륜 결함 특유의 현상)


In [ ]:
# Hilbert envelope (utils 래퍼 사용)
v_filter_env_n = utils.hilbert_envelope(v_filter_n)
v_filter_env_f = utils.hilbert_envelope(v_filter_f)

# DC 성분 제거 후 FFT
f_env_n, A_env_n = utils.fft(v_filter_env_n - np.mean(v_filter_env_n), fs)
f_env_f, A_env_f = utils.fft(v_filter_env_f - np.mean(v_filter_env_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_env_n, A_env_n, color='C0', label='Normal', alpha=0.9)
plt.plot(f_env_f, A_env_f, color='C1', label='Fault (DE IR)', alpha=0.7)

# shaft harmonic (검은색), BPFI harmonic (빨강) — 첫 번째에만 label 달아 legend 중복 방지
for n in range(1, 4):
    plt.axvline(n * F_shaft, color='k', linestyle='dashed', alpha=0.5,
                label='Shaft harmonic' if n == 1 else None)
    plt.axvline(n * DE_BPFI, color='C3', linestyle='dashed', alpha=0.7,
                label='BPFI harmonic' if n == 1 else None)

plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, 600])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- Fault 포락선 스펙트럼에 **BPFI ≈ 160 Hz + 배음(2·BPFI ≈ 320 Hz, 3·BPFI ≈ 480 Hz)** 이 뚜렷하게 나타난다.
- Normal은 같은 주파수대에서 BPFI 피크가 없다.
- **bandpass로 공진 대역을 격리한 덕분에 저주파 진단 성분이 살아남은** 것이다. 만약 원 신호를 그대로 envelope 했다면 저주파 잡음과 구동 성분에 묻혀버렸을 것이다.
- 이게 베어링 진단의 핵심 파이프라인 — **bandpass → Hilbert envelope → FFT**.


---


## 실습 6. 특징 벡터 추출과 Fault/Normal Ratio

지금까지는 그림으로 비교했다. 실제 진단 시스템은 **정량화된 숫자(특징, feature)** 가 필요하다. `utils.feature`는 앞선 파이프라인을 그대로 래핑한다.

| Feature 그룹 | 내용 |
| --- | --- |
| `RMS / Skew / Kurt / CF` | 원 신호의 시간 영역 통계 |
| `Band1` | 원 신호에서 공진 대역 `[3500, 5500] Hz` RMS |
| `RMS_filter / …` | bandpass 후 신호의 시간 영역 통계 |
| `Band1_env, Band2_env, Band3_env` | 포락선 스펙트럼에서 BPFI, 2·BPFI, 3·BPFI 주변 에너지 |

**`band_from_envelope` 설계**

베어링 결함은 슬립·하중 변화에 따라 BPFI가 이론값에서 **±5~10% 흔들린다**. 단일 주파수 bin 한 개만 읽으면 피크가 옆으로 새어 놓칠 수 있다. 그래서 **1배/2배/3배 harmonic 각각에 ±10% 윈도우**를 잡고 그 안 에너지를 적분한다.


In [ ]:
band_filter = [3500, 5500]  # 공진 분석 대역 = 실습 4 와 동일

# BPFI 및 배음 각각 ±10% 윈도우
band_from_envelope = np.array([
    [1 * DE_BPFI * 0.9, 1 * DE_BPFI * 1.1],
    [2 * DE_BPFI * 0.9, 2 * DE_BPFI * 1.1],
    [3 * DE_BPFI * 0.9, 3 * DE_BPFI * 1.1],
])

print('Bandpass filter range:', band_filter)
print('Envelope harmonic bands (Hz):')
print(band_from_envelope)


In [ ]:
feature_n, feature_name = utils.feature(v_n, fs, band_filter, band_from_envelope)
feature_f, feature_name = utils.feature(v_f, fs, band_filter, band_from_envelope)

plt.figure(figsize=(10, 5))
plt.plot(feature_n, '-o', color='C0', label='Normal')
plt.plot(feature_f, '-x', color='C1', label='Fault (DE IR)')
plt.xticks(np.arange(len(feature_name)), feature_name, rotation=60, ha='right')
plt.ylabel('Feature value')
plt.legend()
plt.tight_layout()
plt.show()


**관찰 — 절대값 비교**
- 시간 영역 feature (`RMS`, `Skew`, `Kurt`, `CF`) 는 정상과 결함의 절대값 차이가 크지 않다.
- 반면 **`Band1_env` / `Band2_env` / `Band3_env`** 는 결함에서 훨씬 크게 튄다.
- 단, 절대값 스케일이 feature마다 다르므로 (`Kurt`는 수~수십, `Band*_env`는 0.001 단위) 한 그림에 같이 찍으면 작은 feature가 묻힌다. **비율(ratio)** 로 다시 보자.


In [ ]:
feature_ratio = feature_f / feature_n

plt.figure(figsize=(13, 5))

# Full scale
plt.subplot(1, 2, 1)
plt.plot(feature_ratio, '-o', color='C0', label='Fault / Normal')
plt.axhline(y=1, color='C3', linestyle='--', label='ratio = 1')
plt.xticks(np.arange(len(feature_name)), feature_name, rotation=60, ha='right')
plt.ylabel('Fault / Normal ratio')
plt.title('Full scale')
plt.legend()

# Zoom (0~10) — 시간 feature 상세 확인
plt.subplot(1, 2, 2)
plt.plot(feature_ratio, '-o', color='C0', label='Fault / Normal')
plt.axhline(y=1, color='C3', linestyle='--', label='ratio = 1')
plt.ylim([0, 10])
plt.xticks(np.arange(len(feature_name)), feature_name, rotation=60, ha='right')
plt.ylabel('Fault / Normal ratio')
plt.title('Zoom (0-10)')
plt.legend()

plt.tight_layout()
plt.show()

# 숫자로도 확인
for name, r in zip(feature_name, feature_ratio):
    print(f'{name:<14s}: {r:7.2f}')


**관찰 — 비율 비교 (핵심)**
- **Full scale (왼쪽)**: `Band1_env` (BPFI harmonic 1) 비율이 **수십 배**로 압도적이다. 다른 feature는 모두 1 근처에 붙어 보인다.
- **Zoom 0–10 (오른쪽)**: envelope band ratio는 수십 배, 시간 feature는 1~2 배 수준 — 스케일을 통일하면 시간 feature가 안 보이므로 **zoom이 필요**하다.
- 시간 통계 중에서는 `Kurt` · `CF` (임펄스성 지표) 가 상대적으로 잘 반응하지만, envelope band ratio에는 한참 못 미친다.
- 결론: **Envelope demodulation이 베어링 진단의 핵심 단계**이다. 시간 영역 feature 단독은 분별력이 낮다.


---
